# Part 3: Baseline and model evaluation

Owner: _(name)_

The random walk baseline is the clock every model is timed against. Each model notebook saves its
test-set predictions to `results/<model>_predictions.csv`; this notebook puts them in one table.

### Setup (same cell in every notebook)

Runs in Colab and in Cursor. In Colab it clones the repository first; for a private repository
paste a GitHub token when asked (Settings > Developer settings > Fine-grained tokens, read access
to this repository is enough).

In [ ]:
import sys, pathlib

REPO_URL = "https://github.com/OWNER/sp500-forecasting.git"   # <- set once after the repo exists

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not pathlib.Path("sp500-forecasting").exists() and not pathlib.Path("src/config.py").exists():
    from getpass import getpass
    token = getpass("GitHub token (leave empty if the repo is public): ")
    url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL
    import subprocess
    subprocess.run(["git", "clone", "--quiet", url], check=True)
if IN_COLAB and pathlib.Path("sp500-forecasting").exists():
    import os
    os.chdir("sp500-forecasting")

root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from src import config, common

df = common.load_prices()
print(root)
print(df.index.min().date(), "to", df.index.max().date(), "|", len(df), "trading days")
df.tail()

## 1. The random walk baseline on the test set

In [ ]:
y = common.target_series(df)
train, val, test = common.split(y)

if config.TARGET == "log_return":
    baseline = common.zero_forecast(test)                          # random walk: expected change 0
    drift = common.mean_forecast(test, train.mean())               # random walk with drift
    common.save_predictions("random_walk", test, baseline)
    common.save_predictions("random_walk_drift", test, drift)
else:
    baseline = common.naive_forecast(y).loc[test.index]            # price: tomorrow = today
    common.save_predictions("random_walk", test, baseline)

print("baseline MAE ", round(common.mae(test, baseline), 6))
print("baseline RMSE", round(common.rmse(test, baseline), 6))

## 2. Scorecard: every model in `results/`

In [ ]:
score = common.scorecard()
score

## 3. Prediction plots

In [ ]:
import glob
files = sorted(glob.glob(str(root / "results" / "*_predictions.csv")))
fig, axes = plt.subplots(len(files), 1, figsize=(11, 3 * len(files)), sharex=True)
for ax, f in zip(np.atleast_1d(axes), files):
    p = pd.read_csv(f, parse_dates=[0], index_col=0)
    p["actual"].plot(ax=ax, alpha=0.6, label="actual")
    p["predicted"].plot(ax=ax, alpha=0.9, label="predicted")
    ax.set_title(pathlib.Path(f).stem.replace("_predictions", "")); ax.legend()
plt.tight_layout()

## 4. Reading the table

Say which model beat the baseline, by how much, and whether the gap is big enough to matter
(a difference in the fourth decimal of a daily return is not). A model that does not beat the
random walk is a result too: write it down as one.